# SearchLite-RL final study (Colab, free GPU)
Run the cells **in order**. Order matters: (3) verify the earlier results, (4) the sealed final evaluation (run once), only then (5) the partial-SFT study and (6) the RL-only study. Every code cell is `%%bash` with `set -euo pipefail`, so a failure stops the cell with an error. Nothing is deleted or overwritten; all outputs are written to Google Drive (`MyDrive/SearchLite-RL/`) through symlinks, and re-running a cell resumes (finished stages are skipped).
Cell 2 restarts the runtime once (expected).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%%bash
set -euo pipefail
cd /content
if [ -d Pilot-project ]; then cd Pilot-project && git fetch -q && git checkout -q experiment/evidence-aware-grpo && git pull -q; else git clone -q -b experiment/evidence-aware-grpo https://github.com/aasritha-oql7872/Pilot-project.git; fi
cd /content/Pilot-project
python -c "import trl,peft,transformers" 2>/dev/null && echo "dependencies already installed" || { pip install -q -r requirements.txt; pip uninstall -y -q torchvision torchaudio torchao; echo "INSTALLED: now restart the runtime (Runtime > Restart session) and continue with the next cell"; }
git log --oneline -3

In [ ]:
%%bash
set -euo pipefail
cd /content/Pilot-project
nvidia-smi -L
D=/content/drive/MyDrive/SearchLite-RL
mkdir -p outputs $D/study_sft $D/study_early $D/study_rl_only
for s in study_sft study_early study_rl_only; do ln -sfn $D/$s outputs/$s; done
mkdir -p outputs/main_v2
[ -f outputs/main_v2/adapter/adapter_config.json ] || cp -rn $D/outputs/main_v2/adapter outputs/main_v2/ 2>/dev/null || cp -rn $D/main_v2/adapter outputs/main_v2/ 2>/dev/null || echo "condition A adapter not on Drive: run_sealed.sh will stop (A is needed for the sealed table)"
ls outputs/study_sft
python -m pytest -q 2>&1 | tail -3
echo "## 3. verify earlier reported results against the saved evaluation JSONs"
python -m src.verify_reported outputs/study_sft

In [ ]:
%%bash
set -euo pipefail
cd /content/Pilot-project
# SEALED FINAL EVALUATION. Frozen headline = B (recorded in outputs/study_sft/FROZEN before any sealed eval). Run once.
bash run_sealed.sh 2>&1 | tee -a outputs/study_sft/run_sealed_$(date +%Y%m%d_%H%M).log | tail -60
cat outputs/study_sft/summary_sealed.md

In [ ]:
%%bash
set -euo pipefail
cd /content/Pilot-project
# Partial-SFT study (E0/E1/E2), development only. Stops with exit 3 if no checkpoint has a 20-70% sampled second-search rate.
bash run_early_study.sh 2>&1 | tee -a outputs/study_early/run_$(date +%Y%m%d_%H%M).log | tail -60
cat outputs/study_early/summary_early_dev.md

In [ ]:
%%bash
set -euo pipefail
cd /content/Pilot-project
# RL-only exploration study (F0-F3, optional F4 with RUN_F4=1), development only. Pilots gate the full runs.
bash run_rl_only_study.sh 2>&1 | tee -a outputs/study_rl_only/run_$(date +%Y%m%d_%H%M).log | tail -80
cat outputs/study_rl_only/summary_rl_only_dev.md || true

In [ ]:
%%bash
set -euo pipefail
cd /content/Pilot-project
# Re-run the sampled exploration probes for B/C/D with the CORRECT typed reminder (the earlier probe ignored --answer-format).
S=outputs/study_sft
for N in B C D; do
  [ -f $S/explore_typed_$N.json ] || python -m src.probe explore --config configs/train_sft_grpo.yaml --checkpoint $S/$N/adapter \
      --split twohop_dev --samples 8 --answer-format typed --output $S/explore_typed_$N.json | tail -12
done

In [ ]:
%%bash
set -euo pipefail
cd /content/Pilot-project
git rev-parse HEAD > /content/drive/MyDrive/SearchLite-RL/git_commit_final.txt
cp -n configs/*.yaml /content/drive/MyDrive/SearchLite-RL/study_sft/ || true
echo saved; ls /content/drive/MyDrive/SearchLite-RL